# Создание и заполнение данных БД Postgre

In [1]:
%pip install python-dotenv psycopg2-binary pandas


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\maxpl\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import pandas as pd
import psycopg2
from psycopg2.extras import DictCursor
from dotenv import load_dotenv


# Получение секретов

In [3]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке task3_Docker
dotenv_path = os.path.join(project_root, 'task3_Docker', '.env')

# загружаем переменные окружения из указанного файла
if not os.path.exists(dotenv_path):
    raise FileNotFoundError(f"Файл .env не найден: {dotenv_path}")

load_dotenv(dotenv_path, override=True)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH")

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")


Загруженные данные: USER=vorotynskiy, DB=my_db_vorotynskiy, DB_PORT=5433


# Подключение к базе данных PostgreSQL

In [4]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost",  # если Docker контейнер запущен локально, а ноутбук вне Docker
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    conn.autocommit = False
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")

except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")
    raise


Успешное подключение к базе данных!


In [5]:
# пример запроса
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Версия PostgreSQL: {db_version}")

Версия PostgreSQL: ('PostgreSQL 13.23 (Debian 13.23-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)


In [6]:
# получить список таблиц:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall()
print("\nТаблицы в базе данных:")
for table in tables:
    print(f"- {table[0]}")


Таблицы в базе данных:
- departments
- user_logs


Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры. 

In [9]:
# получить список таблиц:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall()
print("\nТаблицы в базе данных:")
for table in tables:
    print(f"- {table[0]}")


Таблицы в базе данных:
- departments
- user_logs


## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки. 

In [11]:
cursor.execute("""
UPDATE user_logs
SET s_all_avg = REPLACE(s_all_avg, ',', '.'),
    s_course_viewed_avg = REPLACE(s_course_viewed_avg, ',', '.'),
    s_q_attempt_viewed_avg = REPLACE(s_q_attempt_viewed_avg, ',', '.'),
    s_a_course_module_viewed_avg = REPLACE(s_a_course_module_viewed_avg, ',', '.'),
    s_a_submission_status_viewed_avg = REPLACE(s_a_submission_status_viewed_avg, ',', '.');
""")
conn.commit()


In [10]:
cursor.execute('SELECT * FROM user_logs LIMIT 10;')
rows = cursor.fetchall()

print("Первые 10 записей:")
for row in rows:
    print(row)

Первые 10 записей:
(71262, 34527, 6, 9, '9', 4, '4', 0, '0', 0, '0', 0, '0', '3', 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34527, 7, 0, '4,5', 0, '2', 0, '0', 0, '0', 0, '0', '3', 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34527, 8, 0, '3', 0, '1,3333', 0, '0', 0, '0', 0, '0', '3', 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34527, 9, 0, '2,25', 0, '1', 0, '0', 0, '0', 0, '0', '3', 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34527, 10, 0, '1,8', 0, '0,8', 0, '0', 0, '0', 0, '0', '3', 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34527, 11, 1, '1,6667', 1, '0,8333', 0, '0', 0, '0', 0, '0', '3', 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34527, 12, 0, '1,4286', 0, '0,7143', 0, '0', 0, '0', 0, '0', '3', 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34527, 13, 0, '1,25', 0, '0,625', 0, '0', 0, '0', 0, '0', '3', 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34527, 14, 0, '1,1111', 

## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [12]:
conn.rollback() 

In [15]:
cursor.execute('''SELECT COUNT(DISTINCT depart)
               FROM user_logs;''')
count = cursor.fetchone()[0]
print(count)

43


##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводит сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

In [20]:
cursor.execute("""
SELECT d.name, COUNT(DISTINCT u.courseid) as course
FROM user_logs u
JOIN departments d ON u.depart = d.id
GROUP BY d.name;
""")
rows = cursor.fetchall()
print("Кафедра и Количество курсов")
for row in rows:
    print(f"{row[0]} = {row[1]}")

def max_kurs (x):
    return x[1]
print(max(rows, key=max_kurs))

Кафедра и Количество курсов
CC = 19
АиИИ = 19
АСУ = 16
АЭПиМ = 33
БИиИТ = 35
ВИ = 22
ВТиП = 25
ГМДиОПИ = 29
ГМиТТК = 40
ГМУиУП = 35
Дизайна = 20
ДиСО = 53
ИиИБ = 16
ИТМ = 3
ЛиП = 33
ЛиУТС = 36
ЛПиМ = 28
Менеджм. = 20
МиТОДиМ = 28
МиХТ = 42
ПиСЗ = 32
ПиЭММО = 33
ПМиИ = 19
ПОиД = 42
Психол. = 23
ПЭиБЖД = 11
РМПИ = 28
РЯОЯиМК = 17
СРиППО = 14
ТиЭС = 19
ТОМ = 30
ТССА = 21
УиИС = 15
УСиБА = 2
Физики = 16
Физкульт. = 5
Химии = 18
ХОМ = 18
ЦДОМ = 4
ЭиМЭ = 16
Эконом. = 32
ЭПП = 16
ЯиЛ = 20
('ДиСО', 53)


## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




In [21]:
# Задание 4.
# Проверяем, есть ли курсы, за которыми закреплено несколько кафедр.

cursor.execute("""
WITH course_departments AS (
    SELECT DISTINCT
        u.courseid,
        d.name AS department_name
    FROM user_logs u
    JOIN departments d ON u.depart = d.id
    WHERE u.courseid IS NOT NULL
      AND u.depart IS NOT NULL
),
multi_department_courses AS (
    SELECT
        courseid,
        COUNT(DISTINCT department_name) AS department_count
    FROM course_departments
    GROUP BY courseid
    HAVING COUNT(DISTINCT department_name) > 1
)
SELECT COUNT(*)
FROM multi_department_courses;
""")
row = cursor.fetchone()
print(f"Количество курсов, закрепленных за несколькими кафедрами: {row[0]}")

cursor.execute("""
WITH course_departments AS (
    SELECT DISTINCT
        u.courseid,
        d.name AS department_name
    FROM user_logs u
    JOIN departments d ON u.depart = d.id
    WHERE u.courseid IS NOT NULL
      AND u.depart IS NOT NULL
),
multi_department_courses AS (
    SELECT courseid
    FROM course_departments
    GROUP BY courseid
    HAVING COUNT(DISTINCT department_name) > 1
)
SELECT
    cd.courseid,
    STRING_AGG(cd.department_name, ', ' ORDER BY cd.department_name) AS departments
FROM course_departments cd
JOIN multi_department_courses mdc ON cd.courseid = mdc.courseid
GROUP BY cd.courseid
ORDER BY cd.courseid;
""")
rows = cursor.fetchall()

print("Курс - Кафедры, которые совместно преподают курс")
for row in rows:
    print(f"{row[0]} - {row[1]}")



Количество курсов, закрепленных за несколькими кафедрами: 60
Курс - Кафедры, которые совместно преподают курс
71495 - ГМиТТК, ПиСЗ, УиИС
71508 - ПиСЗ, УиИС
71541 - ПиСЗ, УиИС
71547 - ПиСЗ, УиИС
71549 - ГМиТТК, ТиЭС
71571 - ЛиУТС, ПиСЗ, УиИС
71632 - CC, ВТиП
71736 - ГМДиОПИ, ЭиМЭ
71852 - АЭПиМ, ЭПП
71857 - АЭПиМ, ЭПП
71884 - АЭПиМ, ЭПП
71892 - АЭПиМ, ТиЭС, ЭПП
71904 - АЭПиМ, ЭПП
72126 - АЭПиМ, УиИС
72314 - Дизайна, ЛПиМ
72347 - Дизайна, ЛПиМ
72358 - МиХТ, ТОМ
72359 - ЛПиМ, МиХТ, ТОМ
72380 - ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
72392 - ЛПиМ, МиХТ, ТОМ
72402 - ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
72416 - ЛПиМ, МиХТ, ТОМ
72447 - ЛПиМ, МиХТ, ТОМ
72457 - ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
72460 - ЛПиМ, МиХТ, ТОМ
72462 - МиХТ, ТОМ
72469 - МиХТ, ТОМ
72800 - АЭПиМ, ХОМ
72865 - Дизайна, ТОМ
72885 - Дизайна, ТОМ, ХОМ
73574 - CC, ДиСО
73986 - БИиИТ, Эконом.
75810 - ГМДиОПИ, ГМиТТК, РМПИ
75833 - ГМДиОПИ, ГМиТТК, ЛиУТС, РМПИ
75834 - ГМДиОПИ, ГМиТТК, РМПИ
75839 - ГМДиОПИ, ГМиТТК, РМПИ
75849 - ГМДиОПИ, ГМиТТК, РМПИ
76293 - ПиСЗ, Эконо

## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


In [ ]:


cursor.execute("""
SELECT namer_level::text, COUNT(DISTINCT userid) AS student_count
FROM user_logs
WHERE namer_level::text IN ('2', '3', '4', '5')
GROUP BY namer_level::text
ORDER BY namer_level::text;
""")
rows = cursor.fetchall()

print("Оценка - Количество студентов")
for row in rows:
    print(f"{row[0]} - {row[1]}")



Оценка - Количество студентов
2 - 1069
3 - 1884
4 - 3243
5 - 3407


## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за вест период обучения).

In [24]:
cursor.execute(''' 
    SELECT userid
    FROM user_logs
    GROUP BY userid
    ORDER BY MAX(s_all) DESC
    LIMIT 1;
''')
row = cursor.fetchone()
print(f"Студент id {row[0]}")

Студент id 21606


## Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

In [25]:


cursor.execute("""
SELECT
    num_week,
    ROUND(AVG(s_all)::numeric, 2) AS avg_all_events
FROM user_logs
GROUP BY num_week
ORDER BY num_week;
""")
rows = cursor.fetchall()

print("Неделя - Среднее количество всех событий")
for row in rows:
    print(f"{row[0]} - {row[1]}")



Неделя - Среднее количество всех событий
6 - 13.80
7 - 9.62
8 - 8.03
9 - 9.39
10 - 8.21
11 - 10.02
12 - 9.38
13 - 10.01
14 - 9.86
15 - 10.35
16 - 10.29
17 - 10.52
18 - 9.67
19 - 11.11
20 - 14.45
21 - 18.50
22 - 22.49
23 - 22.26
24 - 23.01
25 - 18.22
26 - 8.60
27 - 1.25
28 - 0.09
29 - 0.05


## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников. 

In [26]:


cursor.execute("""
SELECT d.name, COUNT(DISTINCT u.userid) AS excellent_student_count
FROM user_logs u
JOIN departments d ON u.depart = d.id
WHERE u.namer_level::text = '5'
GROUP BY d.name
ORDER BY excellent_student_count DESC, d.name
LIMIT 1;
""")
excellent_row = cursor.fetchone()

print("Кафедра, у которой больше всего отличников:")
print(f"{excellent_row[0]} - {excellent_row[1]}")

cursor.execute("""
SELECT d.name, COUNT(DISTINCT u.userid) AS bad_student_count
FROM user_logs u
JOIN departments d ON u.depart = d.id
WHERE u.namer_level::text = '2'
GROUP BY d.name
ORDER BY bad_student_count DESC, d.name
LIMIT 1;
""")
bad_row = cursor.fetchone()

print("Кафедра, у которой больше всего двоечников:")
print(f"{bad_row[0]} - {bad_row[1]}")



Кафедра, у которой больше всего отличников:
ДиСО - 310
Кафедра, у которой больше всего двоечников:
Эконом. - 72


## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |

In [28]:


import pandas as pd
from IPython.display import display

cursor.execute("""
    WITH weekly_stats AS (
        SELECT
            CASE 
                WHEN name_osno = '1' THEN 'бюджет'
                WHEN name_osno = '2' THEN 'контракт'
                ELSE name_osno
            END AS payment_type,
            num_week AS week_num,
            ROUND(AVG(s_all), 2) AS avg_total_activity,
            ROUND(AVG(s_course_viewed), 2) AS avg_course_views,
            ROUND(AVG(s_q_attempt_viewed), 2) AS avg_attempt_views
        FROM user_logs
        GROUP BY 
            CASE 
                WHEN name_osno = '1' THEN 'бюджет'
                WHEN name_osno = '2' THEN 'контракт'
                ELSE name_osno
            END,
            num_week
    )
    SELECT 
        payment_type,
        week_num,
        avg_total_activity,
        avg_course_views,
        avg_attempt_views
    FROM weekly_stats
    ORDER BY week_num, payment_type;
""")

rows = cursor.fetchall()

columns = [
    "Форма обучения",
    "Номер недели",
    "Средняя активность",
    "Средний просмотр курса",
    "Средний просмотр попыток"
]

df = pd.DataFrame(rows, columns=columns)

display(df)

,Форма обучения,Номер недели,Средняя активность,Средний просмотр курса,Средний просмотр попыток
0,бюджет,6,16.25,5.35,1.00
1,контракт,6,7.54,1.90,0.64
2,бюджет,7,10.32,3.11,0.51
3,контракт,7,7.83,1.94,0.62
4,бюджет,8,8.53,2.40,0.57
5,контракт,8,6.74,1.50,0.53
6,бюджет,9,10.26,2.63,0.83
7,контракт,9,7.18,1.55,0.99
8,бюджет,10,8.84,2.21,0.84
9,контракт,10,6.60,1.23,1.29


In [ ]:

cursor.close()
conn.close()
print("Соединение с БД закрыто.")
